# Homework: Generative AI for Construction Contracts and Drawings
In this homework, we will use generative AI (GPT/HokieAI) to answer questions from a contract, take off quantities from drawings, read door schedules, and analyze a construction document of your own, and compare its outputs with the ground truth.

In every HokieAI step, the notebook gives you a prompt; you send it to HokieAI (https://hokie.ai.vt.edu/), paste HokieAI's output back, and the notebook scores it against the ground truth (the correct output, known for every example).

In [ ]:
#@title ▶ Step 0 · Run me first (1 minute) { display-mode: "form" }
#@markdown Click ▶ and wait for the ✅ line.
import importlib, os, shutil, subprocess, sys
REPO, FOLDER, PKG = "CEM4644", "mp7_prompt_engineering", "aec_prompt"
FOLDERS = ["mp7_prompt_engineering"]          # only this lab folder is downloaded, not the whole course repository

def _git(*args):
    return subprocess.run(["git", "-C", REPO, *args], capture_output=True, text=True).returncode == 0

if os.path.isdir(REPO):                      # a copy is already here: pull the newest course code over it
    if not (_git("sparse-checkout", "set", *FOLDERS)
            and _git("fetch", "-q", "--depth", "1", "origin", "master")
            and _git("reset", "-q", "--hard", "FETCH_HEAD") and _git("clean", "-qfd")):
        shutil.rmtree(REPO, ignore_errors=True)          # broken copy: start again from scratch
if not os.path.isdir(REPO):
    subprocess.run(["git", "clone", "-q", "--depth", "1", "--filter=blob:none", "--sparse", "https://github.com/Haolan-Zhang/CEM4644.git", REPO], check=True)
    subprocess.run(["git", "-C", REPO, "sparse-checkout", "set", *FOLDERS], check=True)
for _m in [m for m in list(sys.modules) if m == PKG or m.startswith(PKG + ".")]:
    del sys.modules[_m]                      # Python caches imported code: drop it, or this cell keeps the old version
importlib.invalidate_caches()
sys.path.insert(0, os.path.abspath(os.path.join(REPO, FOLDER)))
from aec_prompt import lab
lab.setup()


## Part 1 · Answering Questions from a Contract
Twenty-nine construction clauses of the Federal Acquisition Regulation (FAR), the standard terms of U.S. federal construction contracts, and 27 questions about them. Ten questions describe a project situation (a late notice, a withheld payment, a warranty claim), so the answer needs the requirement and its application to the facts. Seven cannot be answered from the text, although most have a common answer outside it. The ground truth is the answer the clauses give; quotes are checked word for word against the clauses.

In [ ]:
#@title ▶ Step 1a · Ask HokieAI About the Contract { display-mode: "form" }
#@markdown Choose a prompt version, run the cell, and follow the instructions. Score every version.
#@markdown Unsupported answer: an answer to one of the questions the clauses do not answer; the correct output says that the clauses do not answer it. Quote found word for word: the quoted sentence appears in the clauses exactly as quoted; a quote that is not found is a misquote.
#@markdown v1 · questions only: the questions and the clauses.
#@markdown v2 · answers with quotes: asks for the sentence of the contract that supports each answer.
#@markdown v3 · only from the text, with quotes: restricts HokieAI to the clauses, asks it to apply the requirement to the facts of each situation, for the clause number and an exact quote, and for NOT IN TEXT when the clauses do not answer.
#@markdown my own prompt: starts from v3; edit the prompt in the prompt box before copying it.
prompt_version = "v1 · questions only" #@param ["v1 · questions only", "v2 · answers with quotes", "v3 · only from the text, with quotes", "my own prompt"]
# The wording the notebook shows (prompts, instructions, results): edit the text between the triple quotes.
# Words in {braces} are filled in by the notebook; **bold** and *italic* work; each line is shown as its own line.
v1_prompt_text = """Answer the questions about the construction contract clauses below.
Reply with one line per question: Q01 | answer

Questions:
{questions}

Contract clauses:
{clauses}"""
v2_prompt_text = """Answer the questions about the construction contract clauses below. For each question, quote the sentence of the contract that supports your answer.
Reply with one line per question: Q01 | answer | exact quote

Questions:
{questions}

Contract clauses:
{clauses}"""
v3_prompt_text = """You are a contract administrator. Answer each question using only the contract clauses below; do not use outside knowledge.
If a question describes a situation, state the requirement and apply it to the facts (dates, amounts, quantities).
Reply with one line per question: Q01 | short answer | FAR clause number | exact quote of the supporting sentence, copied word for word
If the clauses do not answer the question, reply: Q01 | NOT IN TEXT | - | -

Questions:
{questions}

Contract clauses:
{clauses}"""
steps_text = """1. Copy the prompt below.
2. Open {url} (sign in with your VT account), start a new chat, and paste the prompt.
3. Copy HokieAI's output and paste it into the box below.
4. Click *Score HokieAI output* to compare it with the ground truth."""
button_text = """Score HokieAI output"""
reply_text = """Paste HokieAI's full output here."""
prompt_label_text = """prompt"""
reply_label_text = """output"""
result_text = """**{version}**
**Correct answers:** {right} of {n}
**Unsupported answers:** {made_up} of the {unanswerable} questions the clauses do not answer
**Quotes found word for word:** {verified} of {quotes}"""
table_text = """Ground truth vs. HokieAI's output:"""
columns_text = """Question | Ground truth | HokieAI's output | Correct | Quote found in the clauses"""
lab.contract_questions(prompt_version, v1_prompt_text=v1_prompt_text, v2_prompt_text=v2_prompt_text, v3_prompt_text=v3_prompt_text, steps_text=steps_text, button_text=button_text, reply_text=reply_text, prompt_label_text=prompt_label_text, reply_label_text=reply_label_text, result_text=result_text, table_text=table_text, columns_text=columns_text)


In [ ]:
#@title ▶ Step 1b · Compare Prompt Versions { display-mode: "form" }
#@markdown Run the cell after Step 1a. Each bar is the share of questions answered correctly, including NOT IN TEXT for the questions the clauses do not answer.
# The wording the notebook shows (prompts, instructions, results): edit the text between the triple quotes.
# Words in {braces} are filled in by the notebook; **bold** and *italic* work; each line is shown as its own line.
title_text = """Share of correct outputs by prompt version"""
table_text = """All scored outputs so far:"""
empty_text = """No scored outputs yet: run the step above first."""
axis_text = """correct (%)"""
lab.compare("contract", title_text=title_text, table_text=table_text, empty_text=empty_text, axis_text=axis_text)


### 📝 Report question 1
> From Step 1a: Report, for each prompt version, the correct answers, the unsupported answers, and the quotes found word for word.

> Choose one answer that HokieAI took from outside the clauses or misquoted. How could you detect this on a real project, where no ground truth is available?

## Part 2 · Quantity Takeoff from Drawings
Five foundation plans and five gable roofs were drawn for this course. Each comes with estimating questions (CMUs and grout; roof area, underlayment, and shingles), and the ground truth is computed exactly from the drawing's dimensions with standard estimating conventions. HokieAI reads the attached drawing and answers each question; quantities within 1–2 % of the ground truth count as correct.

In [ ]:
#@title ▶ Step 2a · Foundation Plans { display-mode: "form" }
#@markdown Choose a drawing and a prompt version, run the cell, and follow the instructions. Run all three versions on at least two drawings.
#@markdown v1 · task only: the questions and the output format.
#@markdown v2 · step-by-step reasoning: asks HokieAI to list the dimensions it reads and to solve each question step by step.
#@markdown v3 · estimating conventions: adds the estimating conventions used to compute the ground truth.
#@markdown my own prompt: starts from v3; edit the prompt in the prompt box before copying it.
drawing = "Foundation plan 1" #@param ["Foundation plan 1", "Foundation plan 2", "Foundation plan 3", "Foundation plan 4", "Foundation plan 5"]
prompt_version = "v1 · task only" #@param ["v1 · task only", "v2 · step-by-step reasoning", "v3 · estimating conventions", "my own prompt"]
# The wording the notebook shows (prompts, instructions, results): edit the text between the triple quotes.
# Words in {braces} are filled in by the notebook; **bold** and *italic* work; each line is shown as its own line.
v1_prompt_text = """{problem}

{finals}"""
v2_prompt_text = """{problem}

First list every dimension and given value you read from the drawing and the text. Then solve each question step by step: state the formula, substitute the numbers, and compute. Check that each result is reasonable before giving it.
{finals}"""
v3_prompt_text = """{problem}

{conventions}
First list every dimension and given value you read from the drawing and the text. Then solve each question step by step: state the formula, substitute the numbers, and compute. Check that each result is reasonable before giving it.
{finals}"""
finals_text = """End your reply with one final line per question, in this format:
{lines}"""
steps_text = """1. Download {files} with the button below and copy the prompt.
2. Open {url} (sign in with your VT account), start a new chat, attach the image, and paste the prompt.
3. Copy HokieAI's output and paste it into the box below.
4. Click *Score HokieAI output* to compare it with the ground truth."""
button_text = """Score HokieAI output"""
reply_text = """Paste HokieAI's full output here."""
prompt_label_text = """prompt"""
reply_label_text = """output"""
result_text = """**{version}, {drawing}:** {right} of {n} quantities within tolerance of the ground truth."""
table_text = """Ground truth vs. HokieAI's output:"""
columns_text = """Question | Quantity | Ground truth | HokieAI's output | Unit | Correct"""
conventions_text = """Estimating conventions: total wall length = sum of the outside dimensions (no deduction at corners); wall area = length x height; one CMU covers 8 in x 16 in = 0.889 ft²; round CMUs up after adding waste. Grout = CMUs before waste x grout per CMU, converted to CY (27 ft³ = 1 CY), then add grout waste. Cells are 8 in apart, so grouting at S in on center fills 8/S of the cells."""
lab.takeoff(drawing, prompt_version, v1_prompt_text=v1_prompt_text, v2_prompt_text=v2_prompt_text, v3_prompt_text=v3_prompt_text, finals_text=finals_text, steps_text=steps_text, button_text=button_text, reply_text=reply_text, prompt_label_text=prompt_label_text, reply_label_text=reply_label_text, result_text=result_text, table_text=table_text, columns_text=columns_text, conventions_text=conventions_text)


In [ ]:
#@title ▶ Step 2b · Gable Roofs { display-mode: "form" }
#@markdown Choose a drawing and a prompt version, run the cell, and follow the instructions. Run all three versions on at least two drawings.
#@markdown v1 · task only: the questions and the output format.
#@markdown v2 · step-by-step reasoning: asks HokieAI to list the dimensions it reads and to solve each question step by step.
#@markdown v3 · estimating conventions: adds the estimating conventions used to compute the ground truth.
#@markdown my own prompt: starts from v3; edit the prompt in the prompt box before copying it.
drawing = "Gable roof 1" #@param ["Gable roof 1", "Gable roof 2", "Gable roof 3", "Gable roof 4", "Gable roof 5"]
prompt_version = "v1 · task only" #@param ["v1 · task only", "v2 · step-by-step reasoning", "v3 · estimating conventions", "my own prompt"]
# The wording the notebook shows (prompts, instructions, results): edit the text between the triple quotes.
# Words in {braces} are filled in by the notebook; **bold** and *italic* work; each line is shown as its own line.
v1_prompt_text = """{problem}

{finals}"""
v2_prompt_text = """{problem}

First list every dimension and given value you read from the drawing and the text. Then solve each question step by step: state the formula, substitute the numbers, and compute. Check that each result is reasonable before giving it.
{finals}"""
v3_prompt_text = """{problem}

{conventions}
First list every dimension and given value you read from the drawing and the text. Then solve each question step by step: state the formula, substitute the numbers, and compute. Check that each result is reasonable before giving it.
{finals}"""
finals_text = """End your reply with one final line per question, in this format:
{lines}"""
steps_text = """1. Download {files} with the button below and copy the prompt.
2. Open {url} (sign in with your VT account), start a new chat, attach the image, and paste the prompt.
3. Copy HokieAI's output and paste it into the box below.
4. Click *Score HokieAI output* to compare it with the ground truth."""
button_text = """Score HokieAI output"""
reply_text = """Paste HokieAI's full output here."""
prompt_label_text = """prompt"""
reply_label_text = """output"""
result_text = """**{version}, {drawing}:** {right} of {n} quantities within tolerance of the ground truth."""
table_text = """Ground truth vs. HokieAI's output:"""
columns_text = """Question | Quantity | Ground truth | HokieAI's output | Unit | Correct"""
conventions_text = """Estimating conventions: run = half the eave-to-eave distance; rise = run x slope; rake length = sqrt(run² + rise²); net roof area = rake length x eave length x 2 roof planes. Underlayment area = net area + ridge lap (eave length x 1 ft) + rake laps (4/12 ft x rake length x 4 rake edges); rolls = area / (roll length x width), rounded up. Starter course = eave length x exposure x 2 eaves; ridge coverage = eave length x 1 ft; rake waste = 4 rake edges x rake length x 0.25 ft²/ft; gross area = net + starter + ridge + rake waste; shingles = gross area / (3 ft x exposure), rounded up."""
lab.takeoff(drawing, prompt_version, v1_prompt_text=v1_prompt_text, v2_prompt_text=v2_prompt_text, v3_prompt_text=v3_prompt_text, finals_text=finals_text, steps_text=steps_text, button_text=button_text, reply_text=reply_text, prompt_label_text=prompt_label_text, reply_label_text=reply_label_text, result_text=result_text, table_text=table_text, columns_text=columns_text, conventions_text=conventions_text)


In [ ]:
#@title ▶ Step 2c · Compare Prompt Versions { display-mode: "form" }
#@markdown Run the cell after Steps 2a and 2b. Each bar is the share of quantities within tolerance of the ground truth, over all drawings scored with that version.
# The wording the notebook shows (prompts, instructions, results): edit the text between the triple quotes.
# Words in {braces} are filled in by the notebook; **bold** and *italic* work; each line is shown as its own line.
title_text = """Share of correct outputs by prompt version"""
table_text = """All scored outputs so far:"""
empty_text = """No scored outputs yet: run the step above first."""
axis_text = """correct (%)"""
lab.compare("takeoff", title_text=title_text, table_text=table_text, empty_text=empty_text, axis_text=axis_text)


### 📝 Report question 2
> From Step 2c: Report the share of correct quantities for each prompt version, across at least two foundation plans and two roofs.

> For one wrong quantity, find in HokieAI's output where the error came from: a misread dimension, a different estimating convention, or an arithmetic error.

> Why does stating the estimating conventions help more than asking for step-by-step reasoning?

## Part 3 · Reading Schedules on a Drawing Sheet
Five 36 × 24 in drawing sheets, made for this course, each carry the door schedules of several unit types, a window schedule, hardware sets, and door-type elevations. The task: list every door of one unit type. The ground truth is the schedule's own data; a door counts as correct only if all six of its fields are transcribed exactly.

In [ ]:
#@title ▶ Step 3a · Door Schedules { display-mode: "form" }
#@markdown Choose a sheet and a prompt version, run the cell, and follow the instructions. Run all three versions on at least two sheets.
#@markdown v1 · whole sheet: HokieAI receives the whole sheet.
#@markdown v2 · cropped schedule: HokieAI receives only the part of the schedule for the unit type.
#@markdown v3 · cropped schedule and column names: adds the column names and asks for an exact transcription.
#@markdown my own prompt: starts from v3 with both images; edit the prompt in the prompt box before copying it.
sheet = "Door schedule sheet 1 (unit type 1A)" #@param ["Door schedule sheet 1 (unit type 1A)", "Door schedule sheet 2 (unit type 1.1)", "Door schedule sheet 3 (unit type 2A)", "Door schedule sheet 4 (unit type 1.2)", "Door schedule sheet 5 (unit type 1A)"]
prompt_version = "v1 · whole sheet" #@param ["v1 · whole sheet", "v2 · cropped schedule", "v3 · cropped schedule and column names", "my own prompt"]
# The wording the notebook shows (prompts, instructions, results): edit the text between the triple quotes.
# Words in {braces} are filled in by the notebook; **bold** and *italic* work; each line is shown as its own line.
v1_prompt_text = """The attached drawing sheet contains a door schedule. List every door of UNIT TYPE {unit}.
{format}"""
v2_prompt_text = """The attached image is the part of a door schedule for UNIT TYPE {unit}. List every door.
{format}"""
v3_prompt_text = """The attached image is the part of a door schedule for UNIT TYPE {unit}. The schedule's columns are, left to right: MARK, DOOR TYPE, DESCRIPTION, WIDTH, HEIGHT, FRAME, FIRE RATING, HDWR SET, LOCATION. Transcribe every row exactly as written; if a value is not legible, write ?.
{format}"""
format_text = """Reply with one line per door and nothing else, in this format:
mark | door type | width | height | hardware set | location
1 | DF | 3'-0" | 7'-0" | 32 | ENTRY"""
steps_text = """1. Download {files} with the button below and copy the prompt.
2. Open {url} (sign in with your VT account), start a new chat, attach the image, and paste the prompt.
3. Copy HokieAI's output and paste it into the box below.
4. Click *Score HokieAI output* to compare it with the ground truth."""
button_text = """Score HokieAI output"""
reply_text = """Paste HokieAI's full output here."""
prompt_label_text = """prompt"""
reply_label_text = """output"""
result_text = """**{version}, {sheet}:** {right} of {n} doors transcribed exactly ({fields_right} of {fields} fields right)."""
table_text = """Ground truth vs. HokieAI's output:"""
columns_text = """Door | Ground truth | HokieAI's output | Correct"""
lab.schedule(sheet, prompt_version, v1_prompt_text=v1_prompt_text, v2_prompt_text=v2_prompt_text, v3_prompt_text=v3_prompt_text, format_text=format_text, steps_text=steps_text, button_text=button_text, reply_text=reply_text, prompt_label_text=prompt_label_text, reply_label_text=reply_label_text, result_text=result_text, table_text=table_text, columns_text=columns_text)


In [ ]:
#@title ▶ Step 3b · Compare Prompt Versions { display-mode: "form" }
#@markdown Run the cell after Step 3a. Each bar is the share of doors transcribed exactly, over all sheets scored with that version.
# The wording the notebook shows (prompts, instructions, results): edit the text between the triple quotes.
# Words in {braces} are filled in by the notebook; **bold** and *italic* work; each line is shown as its own line.
title_text = """Share of correct outputs by prompt version"""
table_text = """All scored outputs so far:"""
empty_text = """No scored outputs yet: run the step above first."""
axis_text = """correct (%)"""
lab.compare("schedule", title_text=title_text, table_text=table_text, empty_text=empty_text, axis_text=axis_text)


### 📝 Report question 3
> From Step 3b: Report the number of doors transcribed exactly with the whole sheet and with the cropped schedule. Why does cropping change the result?

> Before using a door count from HokieAI in an estimate, what would you check, and how?

## Part 4 · Analyzing Your Own Document
In this part, you will test HokieAI on a construction document of your own choice: a text document of 1–10 pages or a drawing sheet (for example, a specification section, a daily report, a site safety plan, meeting minutes, an RFI, or a submittal from an internship or a public project). You write the questions and the answers you find in the document yourself; your expected answers are the ground truth.

Write at least five questions with short, checkable answers (a number, a date, a name, a material), and include at least one question the document does not answer (expected answer: NOT IN TEXT).

In [ ]:
#@title ▶ Step 4a · Ask HokieAI About Your Document { display-mode: "form" }
#@markdown Write a short name for your document and your questions, one per line, as: question | expected answer. Click **Make prompt** and follow the instructions.
#@markdown The notebook lists HokieAI's answers next to your expected answers; you decide which answers are correct.
#@markdown Then improve the prompt in the prompt box (for example, a role, a required answer format, or how to handle information the document does not give), send it in a new chat, and show the answers again: the output is labeled *edited prompt*.
#@markdown Do not use confidential or personal documents: the document leaves your computer when you attach it to HokieAI.
# The wording the notebook shows (prompts, instructions, results): edit the text between the triple quotes.
# Words in {braces} are filled in by the notebook; **bold** and *italic* work; each line is shown as its own line.
prompt_text = """Answer the questions about the attached document ({document}), using only the document; do not use outside knowledge.
If the document does not answer a question, reply NOT IN TEXT.
Reply with one line per question and nothing else: Q1 | answer

Questions:
{questions}"""
document_label_text = """document"""
document_placeholder_text = """a short name for your document, for example: Daily report, March 3"""
questions_label_text = """questions"""
questions_placeholder_text = """One question per line: question | expected answer
What is the specified concrete strength? | 4,000 psi
Who is the inspector on site? | NOT IN TEXT"""
make_text = """Make prompt"""
too_few_text = """Write at least 5 questions, one per line, as: question | expected answer."""
steps_text = """1. Copy the prompt below.
2. Open {url} (sign in with your VT account), start a new chat, attach your document (or paste its text after the prompt), and paste the prompt.
3. Copy HokieAI's output and paste it into the box below.
4. Click *Show HokieAI's answers* to list them next to your expected answers."""
button_text = """Show HokieAI's answers"""
reply_text = """Paste HokieAI's full output here."""
prompt_label_text = """prompt"""
reply_label_text = """output"""
result_text = """**{version}, {document}**"""
table_text = """Your expected answers (the ground truth) vs. HokieAI's output:"""
columns_text = """Question | Your expected answer | HokieAI's output"""
lab.own_document(prompt_text=prompt_text, document_label_text=document_label_text, document_placeholder_text=document_placeholder_text, questions_label_text=questions_label_text, questions_placeholder_text=questions_placeholder_text, make_text=make_text, too_few_text=too_few_text, steps_text=steps_text, button_text=button_text, reply_text=reply_text, prompt_label_text=prompt_label_text, reply_label_text=reply_label_text, result_text=result_text, table_text=table_text, columns_text=columns_text)


### 📝 Report question 4
> From Step 4a: Describe your document (type, length, and source) and include the tables for the given prompt and your edited prompt. For each prompt, how many of HokieAI's answers are correct, and did it reply NOT IN TEXT where the document does not give the answer?

> What did you change in the prompt, and did it help? Choose one wrong answer and explain from HokieAI's output why it happened.

> Would you rely on HokieAI for this kind of document on a real project, and what would you check first?

### Data and model sources
- Contract clauses: Federal Acquisition Regulation, Part 52 (29 construction clauses), public domain, https://www.acquisition.gov/far.
- Drawings: original practice drawings made for CEM4644 (not for construction).
- Your document: provided by you.
- Generative AI: GPT models via HokieAI (Virginia Tech).